# ML in Production Mindset — Sklearn Refresher

ML engineers must think about models differently than data scientists: versioning, serialization, schema enforcement, performance, and failure modes at production scale.

**Topics:** Pipeline serialization, schema validation, model versioning, performance benchmarking, handling edge cases

In [ ]:
import numpy as np
import pandas as pd
import joblib
import time
import json
import hashlib
import os
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import Dict, List, Optional, Any
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

# ===== Simulate production-like dataset =====
n = 5000
df = pd.DataFrame({
    'age':         np.random.randint(18, 70, n),
    'income':      np.random.lognormal(10.8, 0.7, n),
    'credit_score': np.random.normal(680, 80, n).clip(300, 850),
    'employment':  np.random.choice(['employed', 'self_employed', 'unemployed'], n, p=[0.7, 0.2, 0.1]),
    'loan_purpose': np.random.choice(['home', 'auto', 'personal', 'business'], n, p=[0.3, 0.3, 0.25, 0.15]),
})
# Inject missing values
df.loc[np.random.choice(n, 200), 'income'] = np.nan
df.loc[np.random.choice(n, 100), 'employment'] = np.nan

y = (0.3 * (df.credit_score < 620).astype(int) +
     0.2 * (df.employment == 'unemployed').astype(int) +
     np.random.uniform(0, 0.2, n) > 0.3).astype(int)

X_train, X_test, y_train, y_test = train_test_split(df, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print(f'Positive rate: {y.mean():.2%}')

## 1. Production Pipeline with Schema

In [ ]:
@dataclass
class ModelMetadata:
    """Complete model metadata for versioning."""
    model_name: str
    version: str
    training_date: str
    training_samples: int
    val_auc: float
    feature_columns: List[str]
    numeric_features: List[str]
    categorical_features: List[str]
    hyperparameters: Dict[str, Any]
    data_hash: str  # hash of training data for reproducibility

# Build production pipeline
numeric_features = ['age', 'income', 'credit_score']
categorical_features = ['employment', 'loan_purpose']

numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='unknown')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features),
])

hyperparams = {
    'n_estimators': 100,
    'max_depth': 4,
    'learning_rate': 0.1,
    'subsample': 0.8,
    'min_samples_leaf': 20,
    'random_state': 42,
}

model_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', GradientBoostingClassifier(**hyperparams)),
])

# Train
model_pipeline.fit(X_train, y_train)
val_auc = roc_auc_score(y_test, model_pipeline.predict_proba(X_test)[:, 1])
print(f'Validation AUC: {val_auc:.4f}')

# Create metadata
data_hash = hashlib.md5(pd.util.hash_pandas_object(X_train).values.tobytes()).hexdigest()[:8]
from datetime import datetime
metadata = ModelMetadata(
    model_name='credit-risk-gbm',
    version='2.1.0',
    training_date=datetime.utcnow().isoformat(),
    training_samples=len(X_train),
    val_auc=round(val_auc, 4),
    feature_columns=numeric_features + categorical_features,
    numeric_features=numeric_features,
    categorical_features=categorical_features,
    hyperparameters=hyperparams,
    data_hash=data_hash,
)

print(f'\nModel Metadata:')
print(json.dumps(asdict(metadata), indent=2))

## 2. Model Serialization & Versioning

In [ ]:
# Save model bundle: pipeline + metadata together
model_bundle = {
    'pipeline': model_pipeline,
    'metadata': asdict(metadata),
}

# Save with joblib (handles sklearn objects correctly)
model_dir = Path('models')
model_dir.mkdir(exist_ok=True)
model_path = model_dir / f'credit_risk_v{metadata.version}.pkl'

joblib.dump(model_bundle, model_path, compress=3)
file_size_mb = model_path.stat().st_size / 1024 / 1024
print(f'Model saved: {model_path} ({file_size_mb:.2f} MB)')

# Load and verify
loaded_bundle = joblib.load(model_path)
loaded_pipeline = loaded_bundle['pipeline']
loaded_metadata = loaded_bundle['metadata']

# Verify predictions match
preds_original = model_pipeline.predict_proba(X_test[:5])[:, 1]
preds_loaded   = loaded_pipeline.predict_proba(X_test[:5])[:, 1]
assert np.allclose(preds_original, preds_loaded), 'Serialization error: predictions differ!'
print(f'\nSerializaton verification: ✓ (predictions match exactly)')
print(f'Loaded model version: {loaded_metadata["version"]}')

## 3. Schema Validation & Edge Cases

In [ ]:
class SchemaValidator:
    """Validate prediction requests against the expected schema."""
    
    EXPECTED_COLUMNS = {
        'age':          (int, float),
        'income':       (int, float),
        'credit_score': (int, float),
        'employment':   (str, object),
        'loan_purpose': (str, object),
    }
    
    BOUNDS = {
        'age':          (18, 100),
        'income':       (0, 10_000_000),
        'credit_score': (300, 850),
    }
    
    ALLOWED_CATEGORIES = {
        'employment': {'employed', 'self_employed', 'unemployed'},
        'loan_purpose': {'home', 'auto', 'personal', 'business'},
    }
    
    def validate(self, df: pd.DataFrame) -> Dict[str, List[str]]:
        """Validate DataFrame, return dict of {column: [warnings]}."""
        issues = {}
        
        # Check required columns
        missing_cols = set(self.EXPECTED_COLUMNS.keys()) - set(df.columns)
        if missing_cols:
            issues['schema'] = [f'Missing required columns: {missing_cols}']
            return issues
        
        # Check numeric bounds
        for col, (lo, hi) in self.BOUNDS.items():
            out_of_bounds = df[col].dropna()
            out_of_bounds = out_of_bounds[(out_of_bounds < lo) | (out_of_bounds > hi)]
            if len(out_of_bounds) > 0:
                issues[col] = [f'{len(out_of_bounds)} values out of [{lo}, {hi}]']
        
        # Check categories
        for col, allowed in self.ALLOWED_CATEGORIES.items():
            unknowns = set(df[col].dropna().unique()) - allowed
            if unknowns:
                issues[col] = issues.get(col, []) + [f'Unknown categories: {unknowns}']
        
        return issues

validator = SchemaValidator()

# Test with normal data
issues = validator.validate(X_test[:10])
print(f'Valid data: {len(issues)} issues found')

# Test with problematic data
bad_data = X_test[:5].copy()
bad_data.loc[bad_data.index[0], 'age'] = 150  # out of bounds
bad_data.loc[bad_data.index[1], 'employment'] = 'contractor'  # unknown category
issues = validator.validate(bad_data)
print(f'\nBad data: {len(issues)} issues found:')
for col, msgs in issues.items():
    for msg in msgs:
        print(f'  [{col}] {msg}')

# Production edge cases to handle
print()
print('=== Production Edge Cases ===')
edge_cases = [
    ('All features null', 'Return error — cannot predict without data'),
    ('credit_score=999', 'Validate bounds, log warning, reject'),
    ('income=0', 'Valid (unemployed), handle div-by-zero in debt ratio'),
    ('Unknown employment type', 'OHE handle_unknown=ignore → all zeros (acceptable)'),
    ('age=17', 'Below legal minimum, reject with 422'),
    ('Very large batch (>10K)', 'Chunk into batches, add queue'),
    ('Slow prediction (>500ms)', 'Timeout, return 504, alert SRE team'),
]
for case, handling in edge_cases:
    print(f'  {case:<40} → {handling}')

## 4. Performance Benchmarking

In [ ]:
import timeit
import matplotlib.pyplot as plt

# Benchmark: single vs batch prediction latency
batch_sizes = [1, 10, 50, 100, 500, 1000]
n_repeats = 50
latencies = []
throughputs = []

for bs in batch_sizes:
    X_batch = X_test[:bs]
    times = []
    for _ in range(n_repeats):
        start = time.perf_counter()
        model_pipeline.predict_proba(X_batch)
        times.append(time.perf_counter() - start)
    
    median_ms = np.median(times) * 1000
    p99_ms    = np.percentile(times, 99) * 1000
    tps       = bs / np.median(times)  # predictions per second
    latencies.append((bs, median_ms, p99_ms))
    throughputs.append(tps)
    print(f'Batch={bs:>5} | p50={median_ms:>7.2f}ms | p99={p99_ms:>7.2f}ms | {tps:>8.0f} pred/s')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

bs_list = [l[0] for l in latencies]
axes[0].plot(bs_list, [l[1] for l in latencies], 'b-o', lw=2, label='p50')
axes[0].plot(bs_list, [l[2] for l in latencies], 'r--o', lw=2, label='p99')
axes[0].set_xlabel('Batch Size'); axes[0].set_ylabel('Latency (ms)')
axes[0].set_title('Prediction Latency vs Batch Size')
axes[0].set_xscale('log'); axes[0].legend()

axes[1].plot(bs_list, throughputs, 'g-s', lw=2)
axes[1].set_xlabel('Batch Size'); axes[1].set_ylabel('Predictions per second')
axes[1].set_title('Throughput vs Batch Size')
axes[1].set_xscale('log')

plt.tight_layout(); plt.show()

print(f'\nConclusion: batching dramatically improves throughput!')
print(f'Single prediction: {latencies[0][1]:.1f}ms ({throughputs[0]:.0f} pred/s)')
print(f'Batch-100:         {latencies[3][1]:.1f}ms ({throughputs[3]:.0f} pred/s) — {throughputs[3]/throughputs[0]:.0f}x improvement')